In [2]:
import os
import pandas as pd
from uuid import uuid4
import re
from collections import defaultdict
from typing import Tuple
from tqdm.asyncio import tqdm

In [3]:
from gecodb_compound_parser import perform_umlaut

In [4]:
sp18_path = "../resources/schaefer_pankratz_2018/Concordance.full.csv"
celex_gml_path = "../resources/Celex/german/gml/gml.cd"
celex_gmw_path = "../resources/Celex/german/gmw/gmw.cd"

outdir = "../resources/developed/sp18"

In [5]:
sp18 = pd.read_csv(
    sp18_path,
    sep="\t",
    header=0,
    index_col="Match",
    dtype=defaultdict(lambda: str, LE="int"),   # all str, LE int
    quoting=3   # = csv.QUOTE_NONE, separate by tabs only
)
sp18.head()

,DocURL,DocID,SentID,LC,RC,N2Num,N2Typ,N1Lemma,N1Pl,LE
Match,,,,,,,,,,
Apfelästen,http://www.rattenforum.de/phpbb/viewtopic.php?...,3fc371483b113c1f9bbda743f88603d7014f,11,"Eben , als ich neues Futter und frisches Wasse...",waren einige und ganz vereinzelte an den Käfig...,Pl,Ind,Apfel,U,0
Apfelbad,http://old.glamour.de/glamour/7/2/content/0170...,552f4c720b020f5922a890c09ff8d60c2562,3,Dann spielen Sie mit uns und gewinnen Sie zwei...,relaxen ! Ihr Gutschein ist gültig bis Mai 200...,Sg,Ind,Apfel,U,0
Apfelbauer,http://www.genau-dein-obst.de/apfelblog/page/34,2a56dd938e65d789a0a9cca3015f17acdb16,44,Bei schönstem Sonnenschein nimmt Hannes Panter...,. Auf den Obstplantagen wachsen zum größten Te...,Sg,Ind,Apfel,U,0
Apfelbaum,http://blog.solingen-internet.de/totterblotsch...,fa540ade2bbfec78e9a3cb1a54706cd6da5e,60,Zuweilen gibt es aber auch romantische Momente...,""" für eine Hofschaft mit kleiner Straße nicht ...",Sg,Ind,Apfel,U,0
Apfelbäumchen,http://blog.wiwo.de/chefsache/2009/03/14/der-s...,8cd3866b0dd929e9bc7a40904fddb8ae0460,72,Manche Politiker reden ihre Anliegen ohne Not ...,und seinem diskontierten Wert nennt sich übrig...,Sg,Ind,Apfel,U,0


In [6]:
def umlambda(x: str) -> str:
    x = x.lower()
    x = re.sub("(?<![aeiouy])ae", "ä", x, flags=re.IGNORECASE)
    x = re.sub("(?<![aeiouy])oe", "ö", x, flags=re.IGNORECASE)
    x = re.sub("(?<![aeiouy])ue", "ü", x, flags=re.IGNORECASE)
    return x

In [7]:
gml = pd.read_csv(
    celex_gml_path,
    sep="\\",
    header=None,
    dtype=str,
    usecols=[0, 1],
    names=["id", "lemma"],
    index_col="id"
).dropna()

# keep only nouns
gml = gml[gml["lemma"].str[0].str.isupper()]

gml

,lemma
id,
1,A
2,Ae
4,Aal
6,Aar
7,Aas
...,...
51724,Zytologie
51725,Zytoplasma
51726,Zytosom


In [8]:
gmw = pd.read_csv(
    celex_gmw_path,
    sep="\\",
    header=None,
    dtype=str,
    usecols=[1, 3, 4],
    names=["wordform", "id", "forms"],
    index_col="id"
).dropna()

gmw

,wordform,forms
id,,
35,abbestelle,"1SIE,13SKE"
35,abbestellst,2SIE
35,abbestellen,"13PIE,13PKE,i"
35,abbestellte,"13SIA,13SKA"
35,abbestelltet,"2PIA,2PKA"
...,...,...
27964,Papste,dS
27964,Paepsten,dP
27964,Papstes,gS


In [9]:
# joining will automatically drop non-noun wordforms,
# because they have no lemma in gml
gmwl = gmw.join(gml["lemma"], how="inner")  \
    .reset_index(drop=True)  \
    .set_index("wordform")  \
    .sort_index()

gmwl.index = gmwl.index.map(umlambda)
gmwl["lemma"] = gmwl["lemma"].map(umlambda)
gmwl = gmwl[["lemma", "forms"]]

gmwl

,lemma,forms
wordform,,
a,a,"nP,gP,dP,aP,nS,gS,dS,aS"
aal,aal,"nS,dS,aS"
aale,aal,"nP,gP,aP,dS"
aalen,aal,dP
aales,aal,gS
...,...,...
schöbest kegel,kegelschieben,2SKA/
schöbet kegel,kegelschieben,2PKA/
schöbst kegel,kegelschieben,2SKA/


In [10]:
# Comparing the original Match column values with
# with reconstructions from `parse_sp18_comp()`
# (before lemmatization to NomSg with the help of CELEX2!)
# during development of this module
# revealed 154 mismatches in compounds built from
# six different N1. Manual inspection of these mismatches revealed that
# some compounds with N1s 'Dämon', 'Gerät', 'Geräusch', 'Schloss'
# were spelled alternatively, and that compounds
# with N1s 'Auge' and 'Birne' productively exhibit
# alternation between _+n_ and _-e_ (!).
# See more in 'resources/developed/sp18/leftovers/mismatches.csv'.
# Thus, we will correct
# the spellings of the first four N1s and remove
# all compounds with the last two N1s.

# After these two changes, all reconstructed compounds
# after `parse_sp18_comp()` do match the original Match column
# (before lemmatization to NomSg with the help of CELEX2!).

# correct spellings
for old, new in [
    ("Daemon", "Dämon"),
    ("Geraet", "Gerät"),
    ("Geraeusch", "Geräusch"),
    ("Schloß", "Schloss")
]:
    sp18.index = sp18.index.map(
        lambda x: x.replace(old, new)
        if x.startswith(old) else x
    )

# remove compounds with N1s 'Auge' (N=185) and 'Birne' (N=200)
sp18 = sp18[~sp18["N1Lemma"].isin(["Auge", "Birne"])]

In [11]:
# drop NaN (2 rows only)
sp18 = sp18.dropna()

In [12]:
# extract document metadata from the dataset;
# most probably, we won't need it (also
# many URLs are now down, anyways),
# so by decomposing the dataframe, we can
# reduce processing effort; on the other hand,
# joining the metadata back to the main dataframe
# is easy, so we can always do it later if needed

# first, join the left and right contexts into a single column,
# drop the original columns, and assign a unique id to each context
sp18["context"] = sp18.LC + " " + sp18.index + " " + sp18.RC
sp18.drop(columns=["LC", "RC"], inplace=True)
sp18["context_id"] = [str(uuid4()) for _ in range(len(sp18))]
sp18.head()

,DocURL,DocID,SentID,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id
Match,,,,,,,,,,
Apfelästen,http://www.rattenforum.de/phpbb/viewtopic.php?...,3fc371483b113c1f9bbda743f88603d7014f,11,Pl,Ind,Apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",3d6a88eb-0594-40bd-885a-1f67860d5a64
Apfelbad,http://old.glamour.de/glamour/7/2/content/0170...,552f4c720b020f5922a890c09ff8d60c2562,3,Sg,Ind,Apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,c30c22f0-eb8a-47bd-8b88-c685361f699a
Apfelbauer,http://www.genau-dein-obst.de/apfelblog/page/34,2a56dd938e65d789a0a9cca3015f17acdb16,44,Sg,Ind,Apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,6b91aac6-cd98-4e86-bff1-10378bff7ffd
Apfelbaum,http://blog.solingen-internet.de/totterblotsch...,fa540ade2bbfec78e9a3cb1a54706cd6da5e,60,Sg,Ind,Apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,0941d288-4056-4c6c-8e24-1d2689e91a5e
Apfelbäumchen,http://blog.wiwo.de/chefsache/2009/03/14/der-s...,8cd3866b0dd929e9bc7a40904fddb8ae0460,72,Sg,Ind,Apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,88d61414-a6be-4c1e-9b75-7be0cfcd5266


In [13]:
# now, extract the document metadata
# into a separate dataframe
sp18_meta = sp18[
        ["context_id", "DocID", "DocURL", "SentID"]
    ]   \
    .rename(
        columns={"DocID": "doc_id", "DocURL": "url", "SentID": "sent_n"}
    )   \
    .reset_index(drop=True) \
    .set_index("context_id")    \
    .sort_index()

sp18_meta.head()

,doc_id,url,sent_n
context_id,,,
00030d9c-7d43-46d7-9d6d-35eb072f7ccf,8c506b55cb6a255512a26ca67ac314d96ca1,http://www.rasentraktoren-motoren.de/motoren/r...,13
00062401-efda-456e-86c7-8b4e202fd1a6,2e90821188b8797522e1a0f7901d0069d218,http://www.hmfuture.de/forum/archive/index.php...,53
000c8ca1-41fe-41fd-9674-91b8f0b29da9,502bd1898d78fa2c9857229bc84c520254ba,http://www.ncl-deutschland.de/63.html,52
0018bb87-c68a-4f7f-8e4a-5e754e93ba5d,248b7ee22c0a216a7bfabbb7208870950f84,http://www.bismarck-terrassen.de/newsbz.htm,11
00194697-590a-4763-8fac-a150a512ac45,340259e9762312d12544833d1ba13c69f452,http://devshare.de/topic/25643/positionierung-...,6


In [14]:
# finally, drop the metadata columns
# from the main dataframe
sp18.drop(columns=["DocID", "DocURL", "SentID"], inplace=True)
sp18.head()

,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id
Match,,,,,,,
Apfelästen,Pl,Ind,Apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",3d6a88eb-0594-40bd-885a-1f67860d5a64
Apfelbad,Sg,Ind,Apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,c30c22f0-eb8a-47bd-8b88-c685361f699a
Apfelbauer,Sg,Ind,Apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,6b91aac6-cd98-4e86-bff1-10378bff7ffd
Apfelbaum,Sg,Ind,Apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,0941d288-4056-4c6c-8e24-1d2689e91a5e
Apfelbäumchen,Sg,Ind,Apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,88d61414-a6be-4c1e-9b75-7be0cfcd5266


In [15]:
# First, we need to be able to reconstruct the whole
# compound composition. We need that for three reasons:
# 1. To be able to revert the compounds to NomSg,
# which, in its turn, is required to identify duplicates;
# 2. Beside duplicates removal, we will also be able to
# identify contrastive pairs from the same constituents
# but with pluralic vs. non-pluralic linkers. This would
# be needed to assess the external plural effect
# following the method suggested in [pankratz_2017:20]
# (even though the examples are given for the internal plural effect).
# There is a change that the findings with this approach
# will be different as in [schaefer_pankratz_2018:26] since they
# do not contrast such pairs but rather compare all compounds in Pl forms
# to those in Sg forms.
# 3. Finally, similarly to [shmalts_2026], we would want
# to filter out the constituents not present in CELEX-nouns.

def lemmatize_n2(n2: str, expected_num: str) -> str:
    if n2 not in gmwl.index:
        return n2, False
    n2s = gmwl.loc[n2]
    if isinstance(n2s, pd.Series):  # one row found
        return n2s["lemma"], True
    else:
        if n2s.lemma.nunique() == 1:
            # one variant, ok
            return n2s.lemma.iloc[0], True
        else:
            # try finding a variant
            # that would correspond
            # to the annotated number of N2
            ncode = "P" if expected_num == "pl" else "S"
            matching_n2s = n2s[n2s["forms"].str.contains(ncode)]
            if matching_n2s.lemma.nunique() == 1:
                # one variant, ok
                return matching_n2s.lemma.iloc[0], True
    # otherwise: multiple variants matching the expected number,
    # unreliable parsing
    return n2, False


def parse_sp18_comp(
    comp: str,
    n1: str,
    pl_marker: str,
    has_linker: bool,
    n2_num: str,
    pbar: tqdm
) -> Tuple[str, str, str, bool]:

    pl_gc_template = "_+={pl_marker}_" if "U" in pl_marker else "_+{pl_marker}_"
    maybe_uml = perform_umlaut if ("U" in pl_marker and has_linker) else lambda x: x
    pl_marker = pl_marker.replace("U", "")
    # Schäfer & Pankratz 2018 only consider two
    # contrastive options: zero linker vs. pluralic linker;
    # are other non-pluralic linkers present in the dataset?
    linker = pl_marker if has_linker else ""
    n1_cstem = maybe_uml(n1) + linker
    n2 = re.sub(f"^{n1_cstem}", "", comp)

    # now, take the resulting n2 and lemmatize it to NomSg with the help of CELEX2
    n2, is_lemmatized = lemmatize_n2(n2, expected_num=n2_num)

    # gecodb style
    pl_gc = pl_gc_template.format(pl_marker=pl_marker)
    gecodb = f"{n1}{pl_gc if has_linker else '_'}{n2}"

    pbar.update(1)

    return gecodb, pl_gc, n2, is_lemmatized

In [16]:
sp18.index = sp18.index.map(str.lower)
for col in ["N1Lemma", "N2Num", "N2Typ"]:
    sp18[col] = sp18[col].map(str.lower)
sp18.head()

,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id
Match,,,,,,,
apfelästen,pl,ind,apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",3d6a88eb-0594-40bd-885a-1f67860d5a64
apfelbad,sg,ind,apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,c30c22f0-eb8a-47bd-8b88-c685361f699a
apfelbauer,sg,ind,apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,6b91aac6-cd98-4e86-bff1-10378bff7ffd
apfelbaum,sg,ind,apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,0941d288-4056-4c6c-8e24-1d2689e91a5e
apfelbäumchen,sg,ind,apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,88d61414-a6be-4c1e-9b75-7be0cfcd5266


In [17]:
with tqdm(total=len(sp18), desc="Lemmatizing compounds") as pbar:
    sp18["gecodb"], sp18["n1_pl_marker"], sp18["n2"], sp18["is_lemmatized"] = zip(
        *sp18.apply(
            lambda row: parse_sp18_comp(
                comp=row.name,
                n1=row["N1Lemma"],
                pl_marker=row["N1Pl"],
                has_linker=int(row["LE"]),
                n2_num=row["N2Num"],
                pbar=pbar
            ),
            axis=1
        )
    )

sp18.head()

Lemmatizing compounds: 100%|██████████| 9028/9028 [00:12<00:00, 722.30it/s]


,N2Num,N2Typ,N1Lemma,N1Pl,LE,context,context_id,gecodb,n1_pl_marker,n2,is_lemmatized
Match,,,,,,,,,,,
apfelästen,pl,ind,apfel,U,0,"Eben , als ich neues Futter und frisches Wasse...",3d6a88eb-0594-40bd-885a-1f67860d5a64,apfel_ast,_+=_,ast,True
apfelbad,sg,ind,apfel,U,0,Dann spielen Sie mit uns und gewinnen Sie zwei...,c30c22f0-eb8a-47bd-8b88-c685361f699a,apfel_bad,_+=_,bad,True
apfelbauer,sg,ind,apfel,U,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,6b91aac6-cd98-4e86-bff1-10378bff7ffd,apfel_bauer,_+=_,bauer,True
apfelbaum,sg,ind,apfel,U,0,Zuweilen gibt es aber auch romantische Momente...,0941d288-4056-4c6c-8e24-1d2689e91a5e,apfel_baum,_+=_,baum,True
apfelbäumchen,sg,ind,apfel,U,0,Manche Politiker reden ihre Anliegen ohne Not ...,88d61414-a6be-4c1e-9b75-7be0cfcd5266,apfel_bäumchen,_+=_,bäumchen,True


In [18]:
sp18 = sp18[
        [
            "gecodb", "N1Lemma", "n2", "n1_pl_marker",
            "N2Typ", "N2Num", "LE", "is_lemmatized",
            "context", "context_id"
        ]
    ]   \
    .rename(
        columns={
            "N1Lemma": "n1", "N2Typ": "n1_sem",
            "N2Num": "n2_num", "LE": "is_linker_pl"
        }
    )   \
    .reset_index(drop=True) \
    .set_index("gecodb")    \
    .sort_index()

sp18.sample(5)

,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,is_lemmatized,context,context_id
gecodb,,,,,,,,,
vogel_krankheit,vogel,krankheit,_+=_,ind,pl,0,True,Hierbei erleiden die Tiere normalerweise keine...,76fab7c7-98d3-4a0f-ad2a-284875e510ea
schloss_turm,schloss,turm,_+=er_,ind,sg,0,True,Die Kinder hörten aber nicht auf Frau Kennedy ...,ec7f0f22-3a43-476f-9199-ac60722fd257
mutter_+=_thema,mutter,thema,_+=_,ind,sg,1,True,Dies ist an sich noch kein Hinweis auf Parteil...,8e3753f0-1409-4929-a31d-3366ff80c3fa
hand_schaltung,hand,schaltung,_+=e_,ind,sg,0,True,Die gibt es allerdings wegen insgesamt zu geri...,69ef63f2-627c-4713-91b8-8c02e0372857
zahn_nerv,zahn,nerv,_+=e_,ind,sg,0,True,Interessant ist der Zeitpunkt des Auftretens v...,81b4ed33-5f9d-442f-9aaf-83f1ff0e40dc


In [19]:
# From here on, we will have two versions of the dataset
# because reliable deduplication and detection of contrastive Sg-Pl pairs
# depends on the lemmatization of N2:
# 1. A version with lemmatized compounds only. We will extract the
# contrastive pairs from this version and use
# it for the analysis of the external plural effect.
# The compounds remaining after this extraction
# will be deduplicated and used for the analysis of
# the internal plural effect.
# 2. A joint version with both lemmatized and not lemmatized compounds.
# Will be used for the analysis of the internal plural effect only.
# Caution should be exercised when interpreting the results
# of the internal plural effect analysis based on the joint version.

# not-lemmatized compounds (N=1266)
sp18_nl = sp18[sp18["is_lemmatized"] == False]
sp18_nl.drop(columns=["is_lemmatized"], inplace=True)
sp18_nl.to_csv(
    os.path.join(outdir, "leftovers/not_lemmatized.tsv"),
    sep="\t",
    index=True,
    header=True
)

# joint version (N=9028)
sp18_all = sp18.copy()

# default version: lemmatized compounds only (N=7762)
sp18 = sp18_all[sp18_all["is_lemmatized"] == True]
sp18.drop(columns=["is_lemmatized"], inplace=True)

In [35]:
# extract contrastive Sg-Pl pairs from the lemmatized version
sp18["const_comb"] = sp18["n1"] + "+" + sp18["n2"]
ident_const = sp18[sp18.duplicated(subset="const_comb", keep=False)]    \
    .sort_values("const_comb")
ident_const.to_csv(
    os.path.join(outdir, "external_effect/ident_const.tsv"),
    sep="\t",
    index=True,
    header=True
)
ident_const

,n1,n2,n1_pl_marker,n1_sem,n2_num,is_linker_pl,context,context_id,const_comb
gecodb,,,,,,,,,
apfel_+=_bauer,apfel,bauer,_+=_,ind,pl,1,"Kauf dir ein Apfel , und zahle Applepatente gl...",75dbabe4-e187-42a2-9892-4d58d98cfa79,apfel+bauer
apfel_bauer,apfel,bauer,_+=_,ind,sg,0,Bei schönstem Sonnenschein nimmt Hannes Panter...,6b91aac6-cd98-4e86-bff1-10378bff7ffd,apfel+bauer
apfel_+=_baum,apfel,baum,_+=_,ind,pl,1,Die Verlegung auf diesen Termin aus Witterungs...,d8ea3e35-ffae-4469-a7ac-03fbd73318d4,apfel+baum
apfel_+=_baum,apfel,baum,_+=_,ind,pl,1,Wir fahren an großen Obstgärten vorüber . Die ...,13756563-ed9c-4b93-9e58-39e1eddfa185,apfel+baum
apfel_baum,apfel,baum,_+=_,ind,pl,0,Land - und Forstwirtschaft überziehen große Re...,e6b0fd78-957f-4f1b-8a79-4f8ab70a7bb5,apfel+baum
...,...,...,...,...,...,...,...,...,...
zitat_zitat,zitat,zitat,_+e_,ind,pl,0,Hier fließt also ein Engagement in der Kunst m...,e1617b45-5651-4f86-bcdf-1375d23523b8,zitat+zitat
zitat_+e_zitat,zitat,zitat,_+e_,ind,sg,1,Immerhin werden hier scheinbar auch Menschen g...,f7a759c3-30a0-4e05-ad07-f46823d167ee,zitat+zitat
zitat_zweck,zitat,zweck,_+e_,ind,pl,0,"Ein rechtmäßiger Zitatzweck liegt nur vor , we...",1e825a39-e62b-4ab4-adf9-04b233e6f793,zitat+zweck


In [ ]:
ident_const_sum = {
    const_comb: {
        "pl_form": len(
            group[group["n2_num"] == "pl"]
        ),
        "sg_form": len(
            group[group["n2_num"] == "sg"]
        ),
        "pl_linker": len(
            group[group["is_linker_pl"] == 1]
        ),
        "non_pl_linker": len(
            group[group["is_linker_pl"] == 0]
        )
    }
    for const_comb, group in ident_const.groupby("const_comb")
}

ident_const_sum_df = pd.DataFrame.from_dict(ident_const_sum, orient="index")
ident_const_sum_df.to_csv(
    os.path.join(outdir, "external_effect/ident_const_sum.tsv"),
    sep="\t",
    index=True,
    header=True
)
ident_const_sum_df.head()

,pl_form,sg_form,pl_linker,non_pl_linker
apfel+bauer,1,1,1,1
apfel+baum,5,2,3,4
apfel+berg,1,1,2,0
apfel+bäumchen,0,2,1,1
apfel+dieb,1,1,2,0
